In [522]:
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math

In [523]:
FILE = '../data/raw_data/suggestions.csv'
"""
raw df has continous time series data for each user, no gap between days.
1 sugg.select.utime is Nan: invalid
Number of unique users: 37
"""
df = pd.read_csv(FILE)

# df.dropna(subset=['sugg.select.utime'], inplace=True)
df['uid'] = df['user.index']
df['decision_idx_nogap'] = df['decision.index.nogap']

df['decision_datetime'] = pd.to_datetime(df['sugg.decision.utime'])
df['decision_date'] = df['decision_datetime'].dt.date
df['decision_slot'] = df['sugg.decision.slot']

COLS_DROPPED = ['user.index', 'decision.index', 'decision.index.nogap',
                'sugg.select.utime', 'sugg.select.slot', 'sugg.select.update', 'sugg.tz', 'sugg.gmtoff',
                'sugg.decision.utime', 'sugg.decision.slot',
                'sugg.context.utime', 'sugg.response.utime',
                'jbmins10', 'jbmins40', 'jbsteps40', 'jbmins30', 'jbmins60', 'jbsteps60', 'jbmins90', 'jbsteps90','jbmins120', 'jbsteps120',
                'jbsteps10.zero', 'jbsteps30.zero', 'jbsteps40.zero', 'jbsteps60.zero', 'jbsteps90.zero', 'jbsteps120.zero',
                'jbmins30pre', 'jbmins40pre', 'jbsteps40pre', 'jbmins60pre', 'jbsteps60pre',
                'jbsteps30pre.zero', 'jbsteps40pre.zero', 'jbsteps60pre.zero',
                'gfmins10', 'gfmins30', 'gfmins60', 'gfsteps60', 'gfmins30pre', 'gfmins60pre', 'gfsteps60pre',
                'dec.city', 'response.city', 'response.location.exact', 'response.location.category', 'response.weather.condition', 'response.temperature', 'response.windspeed', 'response.snow', 'recognized.activity.response', 'is.prefetch', 'front.end.application', 'tag.active', 'tag.indoor', 'tag.outdoor', 'tag.outdoor_snow', 'connect', 'response.precipitation.chance', 'recognized.activity', 'interaction.count', 'sugg.device.utime', 'sugg.device.since', 'dec.windspeed', 'dec.snow'
]

df.drop(columns=COLS_DROPPED, inplace=True)

users = df['uid'].unique()
n_users = len(users)
ncols = 6
nrows = math.ceil(n_users / ncols)

df.to_csv('../data/test.csv', index=False)


/var/folders/ck/63g66f6935lgl8mlzt8bpztr0000gn/T/ipykernel_34643/3223986838.py:7: DtypeWarning: Columns (0: send.active, 1: send.sedentary, 2: dec.windspeed, 3: dec.precipitation.chance, 4: response.windspeed, 5: response.precipitation.chance) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(FILE)


In [524]:
df = df.groupby(['uid', 'decision_date']).filter(
    lambda x: x['decision_idx_nogap'].notnull().all()
)

print(len(df))
print(len(df.groupby(['uid', 'decision_date'])))

7864
1605


In [525]:
'''
send:
 - 0: no send
 - 1: sedentary
 - 2: active


when send.active == False and send.sedentary == False, returned.message is always 'donotnotify'
when send == False, returned.message is always 'donotnotify'
when returned.message == 'donotnotify', send == False
'''
df = df.dropna(subset=['decision_slot', 'send.active', 'send.sedentary'])

df['send'] = df['send.active'].astype(int) * 2 + df['send.sedentary'].astype(int)
df['send'].value_counts()

df.drop(columns=['send.active', 'send.sedentary'], inplace=True)

In [526]:
df['steps30'] = df['jbsteps30'].fillna(df['gfsteps30'])
df['steps30pre'] = df['jbsteps30pre'].fillna(df['gfsteps30pre'])

df['steps10'] = df['jbsteps10'].fillna(df['gfsteps10'])

df = df.loc[df['steps30'].notna() & df['steps30pre'].notna()]
df.drop(columns=['gfsteps30', 'gfsteps30pre', 'jbsteps30', 'jbsteps30pre', 'gfsteps10', 'jbsteps10'], inplace=True)

df = df.groupby(['uid', 'decision_date']).filter(
    lambda x: len(x) >= 3
)
df = df.sort_values(['uid', 'decision_date'], ascending=[True, True])

In [527]:
df.loc[df['send'] == 0, 'response'] = 'no_send'
df.fillna({'response': 'no_response'}, inplace=True)

,is.randomized,snooze.status,intransit,avail,send,returned.message,response,dec.location.exact,dec.location.category,dec.weather.condition,dec.temperature,dec.precipitation.chance,uid,decision_idx_nogap,decision_datetime,decision_date,decision_slot,steps30,steps30pre,steps10
0,True,False,False,True,1,Hows your energy level this afternoon A 1015 m...,good,The Songbird Cafe,"restaurant,meal_takeaway,food,point_of_interes...",Partly Cloudy,23.3,0.0,1,0.0,2015-07-22 16:31:53,2015-07-22,2.0,1311.0,0.0,0.0
1,True,False,False,True,2,Its almost the end of the workday How about a ...,good,work,work,Partly Cloudy,24.4,0.0,1,1.0,2015-07-22 18:32:10,2015-07-22,3.0,414.0,418.0,414.0
2,False,False,False,True,0,donotnotify,no_send,home,home,Mostly Cloudy,25.8,0.0,1,2.0,2015-07-22 21:31:48,2015-07-22,4.0,341.0,261.0,140.0
3,True,False,False,True,1,Quick look at the clock If the minute ends in ...,no_response,home,home,Partly Cloudy,25.7,0.0,1,3.0,2015-07-22 23:31:50,2015-07-22,5.0,369.0,2913.0,229.0
4,False,False,False,True,0,donotnotify,no_send,home,home,Clear,11.3,0.0,1,4.0,2015-07-23 09:31:31,2015-07-23,1.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8268,True,False,False,True,2,Have you been sitting all day If you can try n...,no_response,home,home,Overcast,5.0,2.0,37,224.0,2016-01-25 22:27:50,2016-01-25,4.0,88.0,0.0,38.0
8269,False,False,False,True,0,donotnotify,no_send,home,home,Overcast,3.9,11.0,37,225.0,2016-01-26 00:03:36,2016-01-26,5.0,19.0,41.0,19.0
8270,True,False,True,False,0,donotnotify,no_send,MarasRoom on ETSY,"store,point_of_interest,establishment",Overcast,2.2,15.0,37,226.0,2016-01-26 13:01:04,2016-01-26,1.0,564.0,151.0,157.0
8271,True,False,False,True,2,Is now a good time for a stroll through the of...,no_response,work,work,Overcast,1.0,15.0,37,227.0,2016-01-26 16:27:50,2016-01-26,2.0,47.0,85.0,0.0


In [528]:
df['weather'] = df['dec.weather.condition']
df['temp'] = df['dec.temperature']

df['weather'] = df['weather'].replace(
    ['unknown',
     'com.google.appengine.labs.repackaged.org.json.JSONObject.getJSONObject(JSONObject.java:516)',
     'com.google.appengine.labs.repackaged.org.json.JSONObject.<init>(JSONObject.java:179)'],
    pd.NA
)

df['decision_datetime'] = pd.to_datetime(df['decision_datetime'])
df = df.set_index('decision_datetime')

df['temp'] = df.groupby('uid')['temp'].transform(
    lambda x: x.interpolate(method='time').ffill().bfill()
)

# 3. 恢复索引
df = df.reset_index()

# 用同一用户同一天的众数填补
def fill_mode(x):
    mode = x.mode()
    return x.fillna(mode[0] if len(mode) > 0 else pd.NA)

df['weather'] = df.groupby(['uid', 'decision_date'])['weather'].transform(fill_mode)
# 剩余的用前向填充
df['weather'] = df.groupby('uid')['weather'].ffill()

df.drop(columns=['dec.weather.condition', 'dec.temperature'], inplace=True)


In [529]:
# weather_mapping = {
#     # 晴朗
#     'Clear': 'Clear',
#
#     # 多云家族
#     'Partly Cloudy': 'Cloudy',
#     'Scattered Clouds': 'Cloudy',
#     'Mostly Cloudy': 'Cloudy',
#     'Overcast': 'Cloudy',
#
#     # 降雨家族
#     'Rain': 'Rain',
#     'Light Rain': 'Rain',
#     'Thunderstorm': 'Rain',
#     'Light Thunderstorms and Rain': 'Rain',
#     'Light Freezing Rain': 'Rain',
#
#     # 降雪家族
#     'Snow': 'Snow',
#     'Light Snow': 'Snow',
#     'Ice Pellets': 'Snow',
#
#     # 低能见度
#     'Fog': 'Fog',
#     'Haze': 'Fog',
#     'Light Freezing Fog': 'Fog',
# }
weather_mapping = {
    # 晴朗
    'Clear': 'clear',

    # 多云家族
    'Partly Cloudy': 'cloudy',
    'Scattered Clouds': 'cloudy',
    'Mostly Cloudy': 'cloudy',
    'Overcast': 'cloudy',

    # 降雨家族
    'Rain': 'bad',
    'Light Rain': 'bad',
    'Thunderstorm': 'bad',
    'Light Thunderstorms and Rain': 'bad',
    'Light Freezing Rain': 'bad',

    # 降雪家族
    'Snow': 'bad',
    'Light Snow': 'bad',
    'Ice Pellets': 'bad',

    # 低能见度
    'Fog': 'bad',
    'Haze': 'bad',
    'Light Freezing Fog': 'bad',
}

df['weather'] = df['weather'].map(weather_mapping)
assert df['weather'].notna().all(), "有未映射的 weather！"

def bucket_temperature(t):
    """温度分桶（基于体感）"""
    if t < -5:    return 'freezing'    # 极冷, n=277
    elif t < 5:   return 'cold'        # 冷, n=1323
    elif t < 15:  return 'cool'        # 凉爽, n=2133
    elif t < 22:  return 'mild'        # 温和, n=1266
    elif t < 28:  return 'warm'        # 温暖, n=1280
    else:         return 'hot'         # 炎热, n=446

df['temp'] = df['temp'].apply(bucket_temperature)

In [530]:
"""
按 6 类规则分类 dec.location.exact, 然后审查每类里的可疑/边界条目.
"""
import re
import pandas as pd

# ---------- 规则 (优先级从上到下, 命中即归类) ----------
# Note: 对小写名字做 re.search 匹配.
RULES = [
    # 0) Override: 学术名下挂的非学术业务 (必须先于 university/north campus 匹配)
    ("service", [
        r"^university of michigan credit union",   # 60 + 2 + 2
        r"^university of michigan:.*\bmd\b",       # MD doctors hosted by UM
        r"^university of mi.*\bmd\b",
        r"^university of michigan hospital",       # 包括 facilities planning 等
        r"^university of mi.*hospital",
        r"^university of michigan c\.s\. mott",    # children's hospital
        r"^university of michigan division-audiology",
        r"^university of michigan pro$",           # 看着像医疗 ('M-Pro'?), 实际暧昧, 默认归 services
        r"^university of mi.*health",
        r"^university of mi.*neurolgy",
        r"^university of mi.*emr",                 # electronic medical records
        r"^dental school parking$",
    ]),

    ("activity", [
        r"^north campus recreation",               # 48
        r"^university of michigan veteran's memorial",  # 7 (纪念碑, 户外)
        r"^university of michigan museum",
        r"^university of michigan: kelsey museum",
        r"^university of michigan diag$",          # 学校广场, 户外休闲
        r"^university of michigan union billiards",
        # 注: 'School of Music, Theatre and Dance' 是院系 -> Work/Study
        # 注: 'Department of Dance' 是院系 -> Work/Study (通过 Department of 关键词)
        # 注: 'University of Michigan Film' / 'Department of Theatre and Drama' 同理
    ]),

    ("shopping", [
        # 学校相关的停车放到 services, 但学习中心明显是工作/学习
    ]),

    # 1) Home / Rest
    ("home", [
        r"^home$", r"^unknown$",
        r"\bhousing\b", r"\bresidence\b", r"\bdormitory\b", r"\bdorm\b",
        r"^northwood v housing",
        r"\bcampground\b",
        r"\blodging\b",
        r"\bhotel\b", r"\bmotel\b", r"\bresort\b",
        r"^comfort inn metro", r"^the homestead$",
        r"^dibrova camp$", r"^secord lake campground",
        r"alvar aalto baker house", r"ella baker graduate house",
        r"^executive residence", r"^residential college",
        r"^oxford houses", r"^friends lake community",
        r"^park terrace",
        r"^windover womens resort",
    ]),

    # 2) Work / Study (大学/学术 + 该用户的高频工作场所 >=18 次)
    ("work", [
        r"^work$",
        # ---- 先排除掉学术名下挂的 非学术 业务 ----
        # 这些规则用前缀匹配, 不命中此处, 会在后面被对应桶捕获:
        #   'University of Michigan Credit Union' -> Health/Services (credit union)
        #   'University of Michigan: ... MD'      -> Health/Services (MD)
        #   'University of Michigan ... Hospital' -> Health/Services
        #   'North Campus Recreation Building'    -> Leisure/Active
        #   'UofM Parking Yellow Lot'             -> Health/Services (parking)
        # 我们通过让对应桶在 Work/Study 之前先匹配掉它们.
        # 学术
        r"^uofm parking", r"^parking lot",  # 停车场都是为去工作/学习而停
        r"science learning",  # Science Learning Center: ...
        r"\buniversity\b", r"\bcollege\b", r"\bschool\b",
        r"\bacademy\b", r"\bacademic\b",
        r"\bdepartment of\b", r"\bdept\b", r"\binstitute\b",
        r"\blaboratory\b", r"\blab\b(?!or)",  # lab 但不是 labor
        r"\bresearch\b", r"\blibrary\b", r"\balumni\b",
        r"\bprogram\b", r"\bcenter for\b", r"\bstudies\b",
        r"^u-?m\b", r"\bumich\b", r"^univ\b",
        r"^michigan engineering$", r"^michigan sea grant$",
        r"^chrysler center$", r"^pierpont commons$",
        r"^ross academic", r"^rackham", r"^hutchins hall",
        r"^ginsberg center", r"^north campus",
        r"^law school", r"^the career center",
        r"^the newnan", r"^ross school", r"^stephen m\. ross",
        r"^taubman college", r"^a\. alfred taubman",
        r"^fisher college", r"\bdiag\b", r"^techarb$",
        r"^millennium project", r"^icpsr",
        r"^inter-university consortium",
        r"^hutchins hall", r"^smith ryerson center",
        r"^alexander g\. ruthven", r"^dana natural resources",
        r"^robert h\. lurie", r"^marine hydrodynamics",
        r"^lsa\b", r"^gerald r\. ford", r"^ace-mrl",
        r"^aerospace propulsion", r"^macromolecular science",
        r"^sociology -",
        r"\bmuseum studies\b",
        r"^michigan clinical law",
        # 该用户的高频工作场所 (>=22 次以上, 几乎肯定是 work; 18-21 算偶尔光顾的小生意)
        r"^dasa llc$",                  # 214
        r"^adit computer store$",       # 111
        r"^pardee michelle l$",         # 77
        r"^pro shop$",                  # 61
        r"^enlighten merchant",         # 54
        r"^alpha kitchen phone$",       # 49
        r"^klaus & effect$",            # 32
        r"^blandford bonnie$",          # 29
        r"^brembo north america$",      # 27
        r"^customcraft$",               # 26
        # 注: 18-21 次访问的 (Mee Li Shop, Luvlee By Santha, Don Miller Appliance,
        # Sew Successful, Encompass Inc.) 不算工作; 它们会落到 Health/Services
        # (通用 Inc/LLC 兜底), 视作偶尔办事.
    ]),

    # 3) Food & Drink
    ("dining", [
        r"\brestaurant\b", r"\bcafé\b", r"\bcafe\b",
        r"\bbakery\b", r"\bbar\b(?! none)(?!nes)", r"\btavern\b",
        r"\bgrill(?:e)?\b", r"\bbistro\b", r"\bdiner\b",
        r"\bpub\b", r"\bbrewing\b", r"\bbrewery\b", r"\bbrewpub\b",
        r"\bcoffee\b", r"\bpizza\b", r"\bpizzeria\b",
        r"\bburger\b", r"\bsandwich\b", r"\bsteakhouse\b",
        r"\bsushi\b", r"\bdeli\b", r"\bbbq\b", r"\bbarbecue\b",
        r"\bsmokehouse\b", r"\bcreamery\b",
        r"\bdonuts?\b", r"\bbagels?\b", r"\bchocolate\b",
        r"\bcupcake\b", r"\bice cream\b", r"\bicecream\b",
        r"\bkitchen\b(?! phone)", r"\bsubs\b", r"\bcantina\b",
        r"\bcuisine\b", r"\bdining\b",
        # 主流连锁
        r"^panera", r"^subway$", r"^starbucks", r"^mcdonald",
        r"^wendy", r"^taco bell$", r"^chipotle", r"^arby",
        r"^jimmy john", r"^applebee", r"^outback steakhouse",
        r"^red lobster", r"^buffalo wild", r"^cottage inn",
        r"^marco's pizza", r"^big boy", r"^biggby", r"^tim hortons",
        r"^chick-fil-a", r"^krispy kreme", r"^dunkin", r"^domino",
        r"^olga's kitchen", r"^olive garden", r"^cinnabon",
        r"^ihop$", r"^baskin", r"^a&w$", r"^fuddrucker",
        r"^jet's pizza", r"^salsarita", r"^ruby tuesday",
        r"^sonic drive", r"^ben & jerry", r"^einstein bros",
        r"^bruegger", r"^bob evans", r"^hardee", r"^kfc$",
        r"^papa john", r"^papa romano", r"^hungry howie",
        r"^cold stone", r"^qdoba", r"^potbelly",
        r"^burger king", r"^subway", r"^little caesars",
        # 一些没有 keyword 的具体名字
        r"^au bon pain$", r"^bert's cafe", r"^panda chinese",
        r"^new china$", r"^china house$", r"^china gate$",
        r"^takara", r"^kang's korean", r"^guilin", r"^lucky kitchen",
        r"^ken's asian", r"^sushi town", r"^tomato brothers",
        r"^dumpling haus", r"^my thai", r"^pies & pints",
        r"^orchid lane$", r"^thelonious monkfish", r"^the bistro",
        r"^small cheval", r"^shanghai inn", r"^oak street beach food",
        r"^bubble island", r"^bake ann arbor",
        r"^mighty good coffee", r"^espresso royale", r"^mujo café",
        r"^beaudries candy cafe", r"^kirkland and ellis cafe",
        r"^eat well cafe", r"^star's cafe", r"^the daily grind",
        r"^the songbird", r"^bagel factory", r"^bearclaw coffee",
        r"^biggby coffee", r"^coffee beanery", r"^jackson coffee",
        r"^juice generation", r"^cafe ollie$", r"^arbor cafe$",
        r"^bake ann arbor$",
        r"^restaurante bonampak", r"^sabor latino", r"^mexican fiesta",
        r"^mexican town", r"^el azteco", r"^tios mexican",
        r"^camelia's mexican", r"^tortaria", r"^mexico lindo",
        r"^the broken egg", r"^no thai", r"^sava's", r"^palio",
        r"^madras masala", r"^brasserie by lm", r"^city smoker",
        r"^smokehouse 52", r"^grizzly peak", r"^burgerfi",
        r"^mommy's cuisine", r"^the lunch room", r"^the beet box",
        r"^great plains burger", r"^cassel's family", r"^leon's family",
        r"^benny's family", r"^ram's horn", r"^frank's restaurant",
        r"^white grove", r"^carson's american", r"^charley's grilled",
        r"^terrace buffet", r"^suvai palace", r"^bistro 2110",
        r"^pizza house$", r"^pizza box$", r"^pizza", r"^thompson's pizzeria",
        r"^neopapali", r"^luna's pizza", r"^angelo's restaurant",
        r"^toarmina's pizza", r"^cottage inn gourmet pizza",
        r"^pita kabob", r"^amer's mediterranean",
        r"^ahmo's gyros", r"^ali baba",
        r"^ramaki", r"^takashi", r"^pancho",
        r"^victors restaurant", r"^blue water grill", r"^bar none restaurant",
        r"^bowery grille", r"^all star grille", r"^baywatch on the beach",
        r"^crossroads saloon", r"^firebird tavern", r"^box bar",
        r"^st\. andrew's bar", r"^b-line bar", r"^time out lounge",
        r"^rathskeller", r"^gandy dancer", r"^the earle$",
        r"^the earle italian", r"^zingerman's roadhouse", r"^mani osteria",
        r"^kathryn's cheese", r"^kilwin", r"^lansing avenue beer",
        r"^the sports bar$", r"^julianna's restaurant",
        r"^old burdick", r"^old carolina barbecue",
        r"^smokey", r"^lazybones smokehouse", r"^fairway bar",
        r"^hockeytown café", r"^hockeytown cafe",
        r"^blue nile ethiopian", r"^willem's on main",
        r"^rick's american cafe", r"^pierre paul design", # actually company - move
        r"^macrina bakery", r"^core bistro$",
        r"^the algonquin club", r"^ladies' literary club",
        r"^cherry republic", r"^cherry",
        r"^the cupcake station", r"^sweet little paws",
        r"^sweet gem confections", r"^kathleens cookies",
        r"^real baked goods", r"^wolverine bagels",
        r"^café con leche", r"^cafe con leche",
        r"^socarrat", r"^tirami su$", r"^enas sweets", r"^gamma's cakes",
        r"^chocolate drop shop", r"^chill spot$",
        r"^that ice cream place", r"^ice cream renaissance",
        r"^dad's dogs", r"^dairy dan",
        r"^genova pizzeria", r"^bread basket deli",
        r"^arbor brewing", r"^cj's brewing",
        r"^v nightclub", r"^oz nightclub", r"^el amigo night club",
        r"^legends bar", r"^shelly's back room", r"^the ravens club",
        r"^powell's pub", r"^stairway to heaven",
        r"^my belly dancer", r"^getup$", r"^revive$",  # ambiguous; placed here
        r"^renaissance$",
        r"^golden apple$", r"^golden needle upholstery$",  # NO - upholstery is service
        r"^little green apple$",
        r"^bubba's vapor",
        r"^1701 executive cigar",
        r"^seacoast artillery",  # very ambiguous; leave as default
        # 注: 'Suncoast Food Mart' = 便利店, 落到 Shopping (food mart 关键词排除)
        r"^vector inc",  # company - service
        r"^real seafood",
        r"^old english sheeepdog",  # rescue - leisure/services
        r"^the ohio state university",
        r"^cooked",
        r"^amadeus restaurant",
        r"^casey's tavern",
        r"^outback steakhouse",
        r"^mancino's pizza",
        r"^salsarita's fresh cantina",
        r"^panda",
        r"^chick-fil-a$",
        r"^arbor brewing company",
        r"^terrace buffet$",
        r"^kommunity kracker",
        r"^the blue apple",
        r"^cottage inn pizza",
        r"^red lobster$", r"^applebee",
        r"^chipotle mexican grill",
        r"^arby's",
        r"^honey", r"^honey bee", r"^honeybee",
        r"^buca di beppo",
        r"^naughty", r"^naughty time",  # novelty shop actually
        r"^omaha steaks",  # store
        r"^thrifty scot supermarket", # market - shopping
        r"^stadium party shoppe",  # liquor - shopping
        r"^vietshoppe",  # ?
        r"^banana republic",
        r"^liquor(?!\s+control)", r"\bliquor store\b",
        # 注: 'Liquor Control Enforcement Office' 是政府部门, 在 Services 里捕获
        # generic food keyword - 但排除 'food mart' (便利店)
        r"\bfood\b(?!\s+mart)(?!\s+hub)(?!\s+co-op)(?!\s+services)",
        r"^food$",
    ]),

    # 4) Leisure / Active
    ("activity", [
        r"\bpark\b(?! avenue)",
        r"\bnature area\b", r"\bnature center\b", r"\bpreserve\b",
        r"\btrail\b", r"\btrailhead\b",
        r"\brecreation\b", r"\brec\b(?!ord)",
        r"\bgym\b", r"\bfitness\b", r"\byoga\b", r"\bcrossfit\b",
        r"\bboxing\b", r"\bathletic\b", r"\bsports\b",
        r"\bspa\b", r"\bsalon\b", r"\bhair\b",
        r"\bclub\b(?! news)",
        r"\bmuseum\b", r"\bgallery\b", r"\btheatre\b", r"\btheater\b",
        r"\bcinema\b", r"\bcasino\b", r"\bstadium\b",
        r"\bspeedway\b(?!$)",  # 'Lightning Speedway' OK; 'Speedway' 单独 = 加油站
        r"\barena\b", r"\bzoo\b", r"\bbotanic\b",
        r"\bgardens?\b(?!\s+center)(?!\s+&\s+landscape)(?!\s+service)",
        r"^the headlands", r"^belle isle", r"^windsor sculpture",
        r"^fremont street", r"^spirit of music garden",
        r"^campus martius", r"^veterans memorial",
        r"^scarlett-mitchell", r"^arbor hills", r"^gallup park",
        r"^hanover square", r"^mary beth doyle", r"^manchester park",
        r"^mill creek", r"^folkstone", r"^kilburn", r"^chi-bro",
        r"^forsythe", r"^lakelands trail", r"^rose and white",
        r"^fuller park", r"^leslie woods", r"^st\. roch",
        r"^cady street dog", r"^hines park dog", r"^grant bark",
        r"^frisinger", r"^earhart park", r"^brookside",
        r"^south university park", r"^south pond", r"^oakridge",
        r"^willow park", r"^spruce park", r"^sylvan park",
        r"^scheffler", r"^riverside consign",  # consignment, not leisure - move
        r"^foxfire north", r"^clinton park", r"^evergreen park",
        r"^pilgrim park", r"^woodbury park", r"^pier park",
        r"^longshore", r"^varier playground", r"^buhr park",
        r"^palmer field", r"^mitchell field", r"^richmond field",
        r"^mckimmy field", r"^plymouth parkway", r"^cedar bend",
        r"^trinkle marsh", r"^waterford bend",
        r"^sterling state park", r"^secord lake campground",
        r"^petoskey city marina", r"^galien river",
        r"^cass benton hills", r"^terhune pioneer",
        r"^regency at bluffs", r"^city of howell: recreation",
        r"^orchard hills athletic", r"^one on one athletic",
        r"^north campus recreation",
        r"^ymca", r"^la fitness$", r"^snap fitness", r"^life time fitness",
        r"^title boxing", r"^bodyfit$", r"^brooklyn bodyburn",
        r"^smartbodies", r"^curves$", r"^everfit", r"^superslow",
        r"^phiit training", r"^harmony yoga", r"^ita yoga",
        r"^hamburg fitness",
        r"^kozy's cyclery", r"\bbicycle\b",
        r"^lightning speedway", r"^pi kappa phi", r"^delta gamma",
        r"^ann arbor country club", r"^new museum store",
        r"^university of michigan museum",
        r"^kelsey museum", r"^leslie science",
        r"university of michigan diag",
        r"^the interactive house",
        r"^reunion cwt", r"^fit for you",
        r"^bahna wrestling",
        r"^mermaid's casino",
        r"^uptown entertainment", r"^cj snaps",
        r"^i'm bouncie", r"^chiavari chair",
        r"^all saints wedding chapel",
        r"^wylie community pool",
        r"^michigan tent rental",
        r"^the arena", r"^scorekeepers",
        r"^running fit$",
        r"^city wide antiques", r"^mike's antiquary",  # antiques - leisure browsing
        r"^star vintage",
        r"^launch board",
        r"^north end zone",
        r"^tuxedo junction",  # bridal - service-ish; leave to default
        r"^b'z ink tattoo",
        r"^buhr park children",
        r"^michigan flower farm",
        # 'Pixley Funeral Home' / 'Plymouth Nursery' 不在这里, 让它们落到 Services
        # 'Political Graveyard' 不在这里, 让它落到 Services (cemetery/graveyard)
        r"^the algonquin club",
        r"^pi kappa phi",
        r"^pointe fitness",
        r"^red oak", r"^ohio state university",
        r"^the homestead",  # but caught earlier in Home
        r"^chelsea recreation",
        r"^stairway to heaven",  # could be bar; left in food
        r"^v nightclub", r"^oz nightclub",
    ]),

    # 5) Health / Services (含通用商业实体兜底 - 用户去办事的小公司)
    ("service", [
        # 真 health
        r"\bpharmacy\b", r"\bhospital\b", r"\bclinic\b", r"\bmedical\b",
        r"\bhealth\b", r"\b(md|m\.d\.|do|d\.o\.|rph|pharm\.?d\.?|rn|dds)\b",
        r"\bdr\.?\b(?!\s*pepper)", r"\bdoctor\b", r"\bphysician\b",
        r"\bdental\b", r"\bdentist\b", r"\bvision\b",
        r"\boptical\b", r"\beye ?care\b", r"\beye specialists\b",
        r"\bhearing\b", r"\baudiology\b", r"\borthop\b",
        r"\bopthalmology\b", r"\bophthalmology\b",
        r"^cvs", r"^walgreens$", r"^rite aid", r"^kroger pharmacy",
        r"^target pharmacy", r"^costco pharmacy", r"^kmart pharmacy",
        r"^meijer pharmacy", r"^walmart pharmacy",
        r"^haggerty drugs", r"^downriver drugs", r"^choice pharmacy",
        r"^holiday pharmacy", r"^arbor lane pharmacy",
        r"^a a pharma", r"^advanced care pharmacy",
        r"^integrated nonclinical", r"^transgender services",
        r"^lifetime medical", r"^mitchell home medical",
        r"^hart medical", r"^wheelchair seating",
        r"^lice brigade", r"^menuha hospice",
        r"^japanese family health",
        r"^liberty pediatrics", r"^audio help hearing",
        r"^personalized hearing care", r"^primary eye",
        r"^pearle vision", r"^northland eye", r"^shaw eyecare",
        r"^eye care associates", r"^henry ford optimeyes",
        r"^attractive eyewear", r"^tomz optical",
        # 殡仪
        r"\bfuneral\b", r"\bhospice\b", r"\bcrematorium\b",
        r"\bcemetery\b", r"\bgraveyard\b",
        r"^pixley funeral", r"^political graveyard",
        r"^memorial floral",
        # 修车 / 加油
        r"\bauto (?:parts|repair|service|body|sales)\b", r"\bauto$",
        r"\btire\b(?! shop)", r"^firestone", r"^spartan tire",
        r"^speedy auto", r"^schwalbach", r"^safeway tire",
        r"^napa auto", r"^o'reilly auto", r"^carquest",
        r"^source one auto", r"^american surplus auto",
        r"^battery wholesale", r"^interstate all battery",
        r"^k&m tire", r"^goodyear", r"^carter's auto",
        r"^accurate wheel", r"^showdown auto", r"^a & b motors",
        r"^bill crispin chevrolet", r"^victory toyota",
        r"^jaguar of novi", r"^suburban toyota", r"^feldman hyundai",
        r"^bud weiser chevrolet", r"^rochester hills chrysler",
        r"^shenango motors", r"^tri-state auto", r"^zaek's auto",
        r"^victory \/ indian powersports", r"^ziebart",
        r"^jay wolfe toyota", r"^ryder truck",
        r"^u\.s\.a\. trailer", r"^jeep compass$",
        r"^mobile marine repair", r"^michigan cat$",
        r"\bgas\b(?! station)", r"^marathon gas$", r"^speedway$",
        r"^bp$", r"^7-eleven$", r"^sunoco", r"^mobil$",
        r"^marathon pipe line", r"^buckeye pipe",
        r"^schmuckal oil", r"^wolverine truck stop",
        r"^roseville petro mart", r"^boss shop",
        # 金融/邮政/政府
        r"\bcredit union\b", r"\batm\b",
        r"^delta management$", r"^netincome$",
        r"^post office", r"^fedex", r"^ups store",
        r"\blocal_government\b", r"\bsanitation\b",
        r"\bpublic works\b", r"^liquor control enforcement",
        r"\benforcement office\b",
        # 害虫/树木/草坪/家居服务
        r"\bpest control\b", r"\bextermin", r"\btree service\b",
        r"\btree farm\b", r"\bnursery\b", r"\bgreenhouse\b",
        r"\bgarden & landscape\b", r"\blandscape\b",
        r"^star lake greenhouse", r"^north star trees",
        r"^michigan flower farm", r"^huron view tree",
        r"^woody's tree", r"^short's tree", r"^kozminski",
        r"^arend tree", r"^d & l tree", r"^huron view",
        r"^pollums natural resources", r"^national park services",
        r"^water services", r"^redford twp",
        # 摄影/打印
        r"\bphoto\b(?! studio)", r"\bprinting\b", r"\bprint\b",
        r"^kolossos", r"^minute man printer", r"^precision copy",
        r"^victory printing", r"^copy-rite", r"^sunrise screen",
        r"^write the vision", r"^printhero", r"^printwithme",
        r"^graphic visions", r"^scapegoatz", r"^blue ribbon embroidery",
        r"^p c tee", r"^ll graphic design",
        # 室内设计/装修
        r"\binteriors\b", r"\bupholstery\b", r"\bcarpet\b",
        r"\bflooring\b", r"\bhardwood\b", r"\bcabinetry\b",
        r"\bcountertops\b", r"\btile shop\b",
        r"^marjory ruedisale", r"^elizabeth hanley design",
        r"^tracy garfield", r"^leslie knecht", r"^jane wood",
        r"^pierre paul design", r"^cyrus interiors",
        r"^bessenberg bindery", r"^all sewn up",
        r"^slow kitchen post", r"^d & l designs",
        r"^elegant empressions", r"^elegant expressions",
        r"^audrey's alterations", r"^warehouse alteration",
        r"\balterations?\b",
        r"^luvlee by santha",          # 21次 - 美容服务
        r"^sew successful",            # 19次 - 定制印刷服务
        r"^sew much thread",           # 缝纫服务
        r"^marasroom",                 # ETSY 店主, 取/寄货
        r"^chucks upholstery", r"^golden needle upholstery",
        r"^don's custom upholstery", r"^superior fine furnishings",
        r"^merkel furniture", r"^merkel carpet",
        r"^vic's floor", r"^floor specialists", r"^champion flooring",
        r"^carpet 4 less$", r"^area rug cleaning",
        r"^sherwin-williams", r"^teknicolors paints",
        r"^quality painters",
        # 建筑/总包
        r"\bcontracting\b", r"\bbuilders?\b", r"\bconstruction\b",
        r"^stella contracting", r"^stephen lamanen",
        r"^carl drayton", r"^ct custom builders",
        r"^quality home builders", r"^riethmiller lumber",
        r"^fulton construction", r"^castle construction",
        r"^express contracting",
        # 草坪/园艺/景观
        r"^kenny's landscaping", r"^abc asphalt",
        r"^primetime lawn", r"^custom personalized lawn",
        r"^countryside lawn", r"^flora five garden",
        # 维修/家电
        r"\bappliance\b", r"\bplumber\b", r"\bplumbing\b",
        r"\belectrician\b", r"\broofing\b", r"\bhandyman\b",
        r"\bhandy man\b", r"\bfix it\b", r"\brepair\b",
        r"^mr\. appliance", r"^mr\. handyman", r"^handy man today",
        r"^benquin business systems", r"^home services at the home depot",
        # 电信/通讯
        r"\bcellular\b", r"\bwireless\b",
        r"^verizon", r"^sprint", r"^cricket wireless",
        r"^metropcs", r"^t-mobile", r"^boingo wireless",
        r"^authorized metropcs", r"^ranger wireless",
        r"^diamond communications",
        r"^performance networks", r"^d/a central",
        # MLM / 美容上门
        r"^arbonne international", r"^may kay cosmetics",
        r"^mary kay", r"^riddle avon", r"^avon$",
        # 珠宝/花店
        r"\bjewelers?\b", r"\bjewelry\b",
        r"^pandora$", r"^heirloom", r"^j\.b\. robinson",
        r"^grenier's jewelry", r"^heirloom fine jewelry",
        r"^lauren jewelry", r"^la jolla fine", r"^raineri jewelers",
        r"^gold craft jewelers", r"^triple crown watch",
        r"^ismos joyería", r"\bflorist\b", r"\bfloral\b",
        r"^floreria jatziry", r"^happy days flowers",
        r"^magnolia, a fresh", r"^michigan memorial floral",
        r"^floral sense", r"^post gardens",
        r"^orchid macdee$", r"^kroger floral",
        r"^michigan memorial",
        # 兜底: 通用商业实体后缀 (用户去办事的小公司, 不是工作场所)
        r"\bllc\b", r"\binc\b\.?", r"\bcorp\b", r"\bcorporation\b",
        r"\bco\.", r"\bcompany\b", r"\bltd\b",
        r"\bassociates?\b", r"\bconsulting\b", r"\benterprises\b",
        r"\bindustries\b", r"\bsystems\b", r"\bsolutions\b",
        r"\bservices\b", r"\bgroup\b", r"\bholdings\b",
        r"\bllp\b", r"\bpllc\b", r"\bplc\b",
        r"\btechnology\b", r"\btechnologies\b",
    ]),

    # 6) Shopping (兜底中的兜底)
    ("shopping", [
        r"\bstore\b", r"\bmarket\b", r"\bmart\b", r"\bshop\b",
        r"\bshoppe\b", r"\bsupply\b", r"\bsupplies\b", r"\bsupplier\b",
        r"\bgrocery\b", r"\bgrocer\b", r"\bsuper ?market\b",
        r"^kroger$", r"^kroger ", r"^meijer$", r"^meijer ",
        r"^whole foods", r"^trader joe", r"^fred meyer",
        r"^safeway$", r"^costco$", r"^costco ", r"^walmart",
        r"^target$", r"^target ", r"^kmart", r"^kohl's",
        r"^macy", r"^nordstrom", r"^t\.?j\.?\s?maxx",
        r"^dsw\b", r"^payless", r"^famous footwear",
        r"^mast shoes", r"^the walking company",
        r"^dollar general", r"^dollar tree", r"^family dollar",
        r"^big lots", r"^lowe's", r"^the home depot",
        r"^home depot", r"^ikea", r"^bed bath", r"^michaels",
        r"^staples", r"^dick's sporting", r"^mc sports",
        r"^norton sporting", r"^gamestop", r"^big book store",
        r"^pages bookshop", r"^friends book", r"^secondhand prose",
        r"^new renaissance bookshop", r"^our lady of grace bookstore",
        r"^j & d", r"^elvis rodriguez", r"^bivouac",
        r"^petite for her", r"^dressbarn", r"^american apparel",
        r"^lands' end", r"^cato fashions", r"^icing by claire",
        r"^j\.crew$", r"^posh fashion", r"^victoria \+ kane",
        r"^the body shop$", r"^l'occitane", r"^things remembered",
        r"^hudson news", r"^cnbc news", r"^little green apple",
        r"^talulabelle", r"^om super store$",
        r"^macy's espot", r"^classic craze",
        r"^hidden treasures", r"^salvation army store",
        r"^ann arbor thrift", r"^st vincent de paul",
        r"^centralia goodwill",
        r"^hilldale farmers", r"^washtenaw food hub",
        r"^market in the park", r"\bhardware\b",
        r"^naebecks", r"^ace barnes", r"^ward's do it best",
        r"^honor hardware", r"^patio market", r"^family farm",
        r"^heartland", r"^dominick", r"^plum market",
        r"^busch's fresh", r"^golam produce", r"^c j minimarket",
        r"^claudia market", r"^babo market", r"^gregory market",
        r"^kenny's mini", r"^knight's market", r"^street deal",
        r"^thrifty scot", r"^stockbridge town", r"^east valley liquors",
        r"^embassy wine", r"^universal liquor", r"^the green solution",
        r"^johns party", r"^main street party", r"^dannys food",
        r"^suncoast food", r"^campus corner party",
        r"^bulk food$", r"^country corner", r"^bread basket deli",
        r"^frona's pantry", r"^strait to the pantry",
        r"^golden apple$", r"^paradise shops",
        r"^the shell factory", r"^the beaches travelmart",
        r"^family farm & home", r"^heartland marketplace",
        r"^lids$", r"^dockers$", r"^champs sports$",
        r"^sally beauty", r"^bicycle$", r"^bicycle store",
        # 大量低频小生意未列出, 它们会落到 'Other' -> 用兜底规则归 Shopping
    ]),
]


def classify(name: str) -> str:
    if not isinstance(name, str):
        return "home"  # NA defensively
    n = name.lower()
    for bucket, patterns in RULES:
        for p in patterns:
            if re.search(p, n):
                return bucket
    return "shopping"  # 兜底: 剩下的零碎小店都算购物

df['loc'] = df["dec.location.exact"].apply(classify)
df.drop(columns=["dec.location.exact", 'dec.location.category'], inplace=True)


print(len(df))
print(df.isna().sum().to_string())
print(df.columns.to_list())

df.to_csv('../data/data_cleaned.csv', index=False)

6477
decision_datetime           0
is.randomized               0
snooze.status               0
intransit                   0
avail                       0
send                        0
returned.message            0
response                    0
dec.precipitation.chance    0
uid                         0
decision_idx_nogap          0
decision_date               0
decision_slot               0
steps30                     0
steps30pre                  0
steps10                     0
weather                     0
temp                        0
loc                         0
['decision_datetime', 'is.randomized', 'snooze.status', 'intransit', 'avail', 'send', 'returned.message', 'response', 'dec.precipitation.chance', 'uid', 'decision_idx_nogap', 'decision_date', 'decision_slot', 'steps30', 'steps30pre', 'steps10', 'weather', 'temp', 'loc']
